In [1]:
# !pip install tabm -q

In [2]:
import pandas as pd
train = pd.read_csv('/kaggle/input/qaz-letters/train.csv')
test = pd.read_csv('/kaggle/input/qaz-letters/test.csv')

In [3]:
# from category_encoders import TargetEncoder
# import numpy as np
# cat_features = train.select_dtypes(exclude=[np.number]).columns.tolist()
# train[cat_features] = train[cat_features].fillna('')
# test[cat_features] = test[cat_features].fillna('')
# # cat_features.remove('coordinates')
# # cat_features.remove('address')
# # cat_features.remove('label')
# te = TargetEncoder()
# train[cat_features] = te.fit_transform(train[cat_features], train['label'])
# test[cat_features] = te.transform(test[cat_features])

In [4]:
# num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
# num_cols.remove('label')

# train[num_cols] = train[num_cols].fillna(0)
# test[num_cols] = test[num_cols].fillna(0)

In [5]:
X_test = test.copy()#.drop(columns=['name', 'coordinates', 'address'])

In [6]:
X = train.drop(columns=['label'])
y = train['label'].astype(str)

In [7]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch
import numpy as np
# Импорт для метрик многоклассовой классификации
from sklearn.metrics import accuracy_score, f1_score 

def train_tabm(
    X_train, y_train, X_val, y_val, X_test, cat_cols,
    device='cuda', fold=1, LR=1e-3, PATIENCE=5,
    # Параметры TabM
    n_blocks=4,
    d_block=128,
    k=32,
    dropout=0.0,
    attention_dropout=0.0,
    residual_dropout=0.0,
    d_out=None, # Определится как число классов
):
    
    num_feats = X_train.shape[1] - len(cat_cols)
    cat_cardinalities = [int(X_train[col].nunique()) for col in cat_cols]
    
    # 🌟 Изменение 1: Определяем число классов
    if d_out is None:
        num_classes = len(np.unique(y_train))
        d_out = num_classes

    # Масштабирование числовых признаков
    sc = StandardScaler()
    num_cols = [i for i in X_train.columns.tolist() if i not in cat_cols]
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    X_train[num_cols] = sc.fit_transform(X_train[num_cols])
    X_val[num_cols] = sc.transform(X_val[num_cols])
    X_test[num_cols] = sc.transform(X_test[num_cols])

    # Инициализация модели: d_out теперь равно числу классов
    model = TabM.make(
        n_num_features=num_feats,
        cat_cardinalities=cat_cardinalities,
        d_out=d_out, # <--- Изменение: число классов
        n_blocks=n_blocks,
        d_block=d_block,
        k=k,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    # 🌟 Изменение 2: CrossEntropyLoss для многоклассовой классификации
    criterion = torch.nn.CrossEntropyLoss() 

    # 🌟 Изменение 3: Целевые переменные - long, без unsqueeze(1)
    X_train_num = torch.tensor(X_train.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_train_cat = torch.tensor(X_train[cat_cols].values, dtype=torch.long, device=device)
    y_train_t = torch.tensor(np.array(y_train), dtype=torch.long, device=device) # <--- Изменение

    X_val_num = torch.tensor(X_val.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_val_cat = torch.tensor(X_val[cat_cols].values, dtype=torch.long, device=device)
    y_val_t = torch.tensor(np.array(y_val), dtype=torch.long, device=device) # <--- Изменение

    train_ds = TensorDataset(X_train_num, X_train_cat, y_train_t)
    val_ds = TensorDataset(X_val_num, X_val_cat, y_val_t)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    best_val_score = -float('inf') # Ищем максимум (Accuracy или F1)
    best_model_state = None
    best_val_preds_probs = None

    patience = 0
    for epoch in range(1, 101):
        # --- Тренировка ---
        model.train()
        for xb_num, xb_cat, yb in train_loader:
            optimizer.zero_grad()
            if len(cat_cols) == 0:
                xb_cat = None
            
            # preds_k - логиты [batch_size, k, num_classes]
            preds_k = model(xb_num, xb_cat) 
            
            # Усредняем по k: [batch_size, num_classes]
            logits_mean = preds_k.mean(dim=1) 
            
            # Loss вычисляется на логитах [N, C] и целевой переменной [N]
            loss = criterion(logits_mean, yb) 
            
            loss.backward()
            optimizer.step()

        # --- Валидация ---
        model.eval()
        all_logits = []
        all_true = []
        with torch.no_grad():
            for xb_num, xb_cat, yb in val_loader:
                if len(cat_cols) == 0:
                    xb_cat = None
                
                preds_k = model(xb_num, xb_cat)
                logits_mean = preds_k.mean(dim=1) 
                
                all_logits.append(logits_mean.cpu().numpy())
                all_true.append(yb.cpu().numpy())
                
        all_logits = np.concatenate(all_logits) # [N, num_classes]
        all_true = np.concatenate(all_true)     # [N]
        
        # 🌟 Изменение 4: Преобразование логитов в предсказанные классы
        # Предсказанные метки - индекс с максимальным логитом
        all_preds_classes = np.argmax(all_logits, axis=1) 
        
        # Метрика: Accuracy и Weighted F1-score
        val_acc = accuracy_score(all_true, all_preds_classes)
        # Используем weighted F1 для учета возможного дисбаланса классов
        val_f1_w = f1_score(all_true, all_preds_classes, average='weighted') 
        
        print(f"Epoch {epoch}: TabM val_loss = {loss.item():.4f}, TabM val_acc = {val_acc:.4f}, TabM val_f1_weighted = {val_f1_w:.4f}")

        # Сохраняем модель по F1-score
        if val_f1_w > best_val_score: 
            best_val_score = val_f1_w
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            
            # Для предсказаний на валидации сохраняем вероятности
            best_val_preds_probs = torch.softmax(torch.tensor(all_logits), dim=1).numpy()
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f'Early stopping on epoch {epoch}')
                break

    print(f"Fold {fold + 1} TabM best_val_f1_weighted = {best_val_score:.4f}")

    # 🌟 Предсказание на тестовом наборе
    model.load_state_dict(best_model_state)
    model.eval()

    X_test_num = torch.tensor(X_test.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_test_cat = torch.tensor(X_test[cat_cols].values, dtype=torch.long, device=device)
    test_ds = TensorDataset(X_test_num, X_test_cat)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    tabm_test_probs = []
    with torch.no_grad():
        for xb_num, xb_cat in test_loader:
            if len(cat_cols) == 0:
                xb_cat = None
            
            preds_k = model(xb_num, xb_cat) # Логиты [batch_size, k, num_classes]
            logits_mean = preds_k.mean(dim=1) # [batch_size, num_classes]
            
            # 🌟 Изменение 5: Преобразование логитов в вероятности (Softmax)
            probs = torch.softmax(logits_mean, dim=1)
            tabm_test_probs.append(probs.cpu().numpy())

    # Возвращаем вероятности для каждого класса: [N, num_classes]
    tabm_test_probs = np.concatenate(tabm_test_probs, axis=0) 

    # Возвращаем вероятности для каждого класса
    return tabm_test_probs, best_val_preds_probs, best_val_score

In [8]:
X.columns = [col.replace('[','_').replace(']','_').replace('<','lt_').replace('>','gt_') for col in X.columns]
X_test.columns   = [col.replace('[','_').replace(']','_').replace('<','lt_').replace('>','gt_') for col in X.columns]

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)
y = le.transform(y)

In [10]:
n_classes = len(np.unique(y))

In [11]:
y = pd.DataFrame(y)

In [12]:
# Обучение моделей
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, f1_score
# from tabm import TabM
from sklearn.metrics import mean_absolute_error, accuracy_score
import torch
from sklearn.ensemble import RandomForestClassifier

# Инициализация массива для предсказаний на тестовых данных
test_preds = np.zeros((len(X_test), n_classes))
lgb_test_preds = np.zeros((len(X_test), n_classes))
tabm_test_preds = np.zeros((len(X_test), n_classes))
xgb_test_preds = np.zeros((len(X_test), n_classes))
cb_test_preds = np.zeros((len(X_test), n_classes))
rf_test_preds = np.zeros((len(X_test), n_classes))


lgb_params = {
    'n_estimators': 3000,
    'max_depth': 2,
    'learning_rate': 0.06,
    'num_leaves': 60,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'device': 'gpu',
    'early_stopping_round': 100,
    'random_state': 42,
}

cb_params = {
    'iterations': 5000,
    'max_depth': 8,
    'learning_rate': 0.03,
    'task_type': 'GPU',
    'early_stopping_rounds': 100,
    'eval_metric': 'Accuracy',
    'random_state': 1905
}


xgb_params = {
    'n_estimators': 1000,
    'max_depth': 10,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'device': 'gpu',
    # 'early_stopping_rounds': 200,
    # 'eval_metric': 'auc',
    'seed': 42,
}


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for i, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print('#'*15, i+1, '#'*15)
    X_train, X_val, y_train, y_val = X.iloc[train_idx], X.iloc[val_idx], y.iloc[train_idx], y.iloc[val_idx]

    # model_cb = CatBoostClassifier(**cb_params)
    # model_cb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    # cb_test_preds += model_cb.predict_proba(X_test[X.columns.tolist()]) / 5

    # model_xgb = XGBClassifier(**xgb_params)
    # model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    # xgb_test_preds += model_xgb.predict_proba(X_test[X.columns.tolist()]) / 5

    # model_lgb = LGBMClassifier(**lgb_params)
    # model_lgb.fit(
    #     X_train, y_train,
    #     eval_set=[(X_val, y_val)],
    # )
    # # val_preds3 = model_lgb.predict_proba(X_val)
    # # val_preds += val_preds3 / 2
    # lgb_test_preds += model_lgb.predict_proba(X_test[X.columns.tolist()]) / 5

    # tabm_preds, tabm_val_preds, score = train_tabm(
    #     X_train, y_train, 
    #     X_val, y_val, 
    #     X_test, [], 
    #     device='cuda', fold=i, 
    #     LR=2e-3, n_blocks=4,
    #     d_block=256, k=32,
    #     dropout=0.2, PATIENCE=20,
    #     d_out = n_classes
    #     )
    
    # tabm_test_preds += tabm_preds / 5
    # val_preds += tabm_val_preds / 3
    # score = accuracy_score(np.argmax(val_preds), y_val)
    # print(f'Score: {score:.4f}. LB Score: {(1/(1+score)):.4f}')
    # scores.append(score)

    # model_rf = RandomForestClassifier(**rf_params)
    # model_rf.fit(X_train, y_train)
    # rf_test_preds += model_rf.predict_proba(X_test[X.columns.tolist()])


# print('Mean Score:', round(np.mean(scores), 4), 'Mean LB score:', round(1/(1+np.mean(scores)), 4))

############### 1 ###############
############### 2 ###############
############### 3 ###############
############### 4 ###############
############### 5 ###############


In [13]:
# from sklearn.ensemble import RandomForestClassifier
# rf_params = {
#     'n_estimators': 1000,         # Количество деревьев в лесу (по умолчанию 100)
#     'max_depth': None,            # Максимальная глубина дерева (по умолчанию None, что означает, что узлы будут расширяться до максимума)
#     'min_samples_split': 2,       # Минимальное количество образцов для разбиения узла
#     'min_samples_leaf': 1,        # Минимальное количество образцов в листьях
#     'max_features': 'auto',       # Количество признаков, которые будут использоваться для поиска лучшего разделения
#     'bootstrap': True,            # Использование бутстрэпа при создании деревьев
#     'random_state': 42,           # Случайное начальное состояние для воспроизводимости
#     'n_jobs': -1,                 # Количество потоков, используемых для обучения (по умолчанию -1 использует все доступные ядра)
#     'class_weight': None,         # Взвешивание классов (если данные несбалансированы, можно указать 'balanced')
#     'oob_score': False            # Оценка точности модели на выборке вне бутстрэпа (out-of-bag)
# }

# model_rf = RandomForestClassifier(**rf_params)
# model_rf.fit(X, y)
# rf_test_preds = model_rf.predict_proba(X_test[X.columns.tolist()])

In [14]:
import numpy as np
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

# Вычисление весов для классов
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights_dict = dict(zip(np.unique(y), class_weights))

# Параметры модели
catboost_params = {
    'iterations': 1000,             # Количество итераций (деревьев)
    'depth': 8,                     # Глубина деревьев
    'learning_rate': 0.1,           # Коэффициент обучения
    'loss_function': 'MultiClass',  # Многоклассовая классификация
    'custom_metric': ['Accuracy'],  # Метрики для оценки
    'cat_features': [],             # Если есть категориальные признаки, их надо передать в список
    'random_state': 42,             # Случайное начальное состояние для воспроизводимости
    'task_type': 'GPU',             # Использование GPU
    'class_weights': class_weights_dict  # Передаем веса классов
}

# Создание модели CatBoost с учетом балансировки классов
model_catboost = CatBoostClassifier(**catboost_params)

# Обучение модели
model_catboost.fit(X, y, verbose=100)

# Прогнозирование на тестовых данных
catboost_test_preds = model_catboost.predict_proba(X_test[X.columns.tolist()])


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0:	learn: 3.1600847	total: 13.6s	remaining: 3h 45m 58s
100:	learn: 0.1570120	total: 1m 1s	remaining: 9m 3s
200:	learn: 0.0790907	total: 1m 38s	remaining: 6m 32s
300:	learn: 0.0566497	total: 2m 15s	remaining: 5m 13s
400:	learn: 0.0417187	total: 2m 51s	remaining: 4m 16s
500:	learn: 0.0320789	total: 3m 28s	remaining: 3m 28s
600:	learn: 0.0256097	total: 4m 6s	remaining: 2m 43s
700:	learn: 0.0210109	total: 4m 43s	remaining: 2m
800:	learn: 0.0175952	total: 5m 20s	remaining: 1m 19s
900:	learn: 0.0149867	total: 5m 57s	remaining: 39.3s
999:	learn: 0.0129107	total: 6m 34s	remaining: 0us


In [15]:
# from sklearn.linear_model import LogisticRegression

# logreg_params = {
#     'penalty': 'l2',              # Регуляризация L2 (по умолчанию)
#     'C': 1.0,                     # Коэффициент регуляризации (по умолчанию 1.0)
#     'solver': 'liblinear',        # Метод оптимизации (liblinear подходит для маленьких и средних наборов данных)
#     'max_iter': 100,              # Максимальное количество итераций для сходимости
#     'random_state': 42,           # Случайное начальное состояние для воспроизводимости
#     'n_jobs': -1                  # Количество потоков для вычислений
# }

# model_logreg = LogisticRegression(**logreg_params)
# model_logreg.fit(X, y)
# logreg_test_preds = model_logreg.predict_proba(X_test[X.columns.tolist()])


In [16]:
test_preds = catboost_test_preds# + logreg_test_preds# + lgb_test_preds
preds = np.argmax(test_preds, axis=1)

In [17]:
preds = le.inverse_transform(preds)
submit = pd.read_csv('/kaggle/input/qaz-letters/sample_submission.csv')
submit['label'] = preds
submit.to_csv('qaz_letters_cb_class_balance.csv', index=False)